In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
def countFrames(video):
    cap = cv2.VideoCapture(video)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return frames

In [ ]:
def collectVideos(root):
    videos = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.upper().endswith(".MOV"):
                videos.append(os.path.join(dirpath, f))
    return videos

In [ ]:
def identifyPitcher(keypoints, frame_shape):
    if len(keypoints) == 1:
        return 0
    h, w = frame_shape[:2]
    expected_x = 0.5 * w
    idx, score = 0, float("-inf")

    for i, pts in enumerate(keypoints):
        xs, ys = pts[:, 0], pts[:, 1]
        x_diff = torch.max(pts[:, 0]) - torch.min(pts[:, 0])
        y_diff = torch.max(pts[:, 1]) - torch.min(pts[:, 1])
        hori = 1 - torch.abs(torch.nanmean(xs) - expected_x) / w
        if (s := 10 * x_diff * y_diff + hori) > score:
            idx, score = i, s
    return idx

In [ ]:
def detect_glove_yolo(frame, glove_model, pitcher_box=None):
    results = glove_model(frame, conf=0.05, verbose=False)[0]
    if results.boxes is None or len(results.boxes) == 0:
        return None
    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()
    glove_boxes = [b for b, c in zip(boxes, classes) if c == 0]
    if not glove_boxes:
        return None
    if pitcher_box is None:
        return list(map(int, glove_boxes[0]))
    px1, py1, px2, py2 = pitcher_box
    pitcher_center = ((px1 + px2) / 2, (py1 + py2) / 2)
    def dist(box):
        x1, y1, x2, y2 = box
        glove_center = ((x1 + x2) / 2, (y1 + y2) / 2)
        return np.linalg.norm(np.array(glove_center) - np.array(pitcher_center))
    best_box = min(glove_boxes, key=dist)
    return [int(v) for v in best_box]

In [ ]:
def getThrowingArm(keypoints):
    rel_x_right = keypoints[:, 10, 0] - keypoints[:, 6, 0]
    rel_x_left  = keypoints[:, 9, 0] - keypoints[:, 5, 0]

    for arr in [rel_x_right, rel_x_left]:
        if np.any(np.isnan(arr)):
            valid = ~np.isnan(arr)
            arr[:] = np.interp(np.arange(len(arr)), np.where(valid)[0], arr[valid])

    range_right = np.nanmax(rel_x_right) - np.nanmin(rel_x_right)
    range_left  = np.nanmax(rel_x_left)  - np.nanmin(rel_x_left)
    return "right" if range_right > range_left else "left"

In [ ]:
def getForwardDirection(rel_x):
    start = int(0.1 * len(rel_x))
    end   = int(0.9 * len(rel_x))
    if end <= start:
        return 1
    start_mean = np.nanmean(rel_x[:start])
    end_mean   = np.nanmean(rel_x[end:])
    return 1 if end_mean > start_mean else -1

In [ ]:
def findPitchStart(keypoints):
    arm = getThrowingArm(keypoints)
    shoulder_idx, wrist_idx = (6, 10) if arm == "right" else (5, 9)
    shoulder_x = keypoints[:, shoulder_idx, 0]
    wrist_x    = keypoints[:, wrist_idx, 0]
    rel_x = wrist_x - shoulder_x

    if np.any(np.isnan(rel_x)):
        valid = ~np.isnan(rel_x)
        rel_x = np.interp(np.arange(len(rel_x)), np.where(valid)[0], rel_x[valid])
    direction = getForwardDirection(rel_x)
    signed_rel = direction * rel_x

    kernel = np.ones(window := 5) / window
    signed_rel_smooth = np.convolve(signed_rel, kernel, mode="same")
    if len(signed_rel_smooth) <= (search_start := 30):
        return 0
    min_idx = np.argmin(signed_rel_smooth[search_start:]) + search_start

    range_after = np.max(signed_rel_smooth[min_idx:]) - np.min(signed_rel_smooth[min_idx:])
    threshold = 0.05 * range_after
    for i in range(min_idx + 1, len(signed_rel_smooth)):
        if signed_rel_smooth[i] > signed_rel_smooth[min_idx] + threshold:
            if i + 3 < len(signed_rel_smooth) and all(
                signed_rel_smooth[j] > signed_rel_smooth[j-1] for j in range(i+1, i+4)
            ):
                return i
            return i
    return min_idx

In [ ]:
def processVideo(pose_model, glove_model, video, savedir):
    cap = cv2.VideoCapture(video)
    frame_idx, kp_seq = 0, []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        res = pose_model(frame, verbose=False)[0]
        if not (res.keypoints and len(persons := res.keypoints.xy)):
            frame_idx += 1
            continue
        idx = identifyPitcher(persons, frame.shape)
        person_kpts = persons.cpu().numpy()
        target_kpts = person_kpts[idx]
        try:
            x_min, y_min = np.min(target_kpts, axis=0)
            x_max, y_max = np.max(target_kpts, axis=0)
            pitcher_box = [x_min, y_min, x_max, y_max]
        except:
            pitcher_box = None
        
        glove_box = detect_glove_yolo(frame, glove_model, pitcher_box)
        if glove_box is None: glove_box = [-1, -1, -1, -1]
        
        kp_seq.append({"keypoints": target_kpts, "glove_box": glove_box})
        frame_idx += 1

    cap.release()
    if not kp_seq:
        print("[ERR] No keypoints extracted")
        return
        
    # --- AUTOMATIC TRUNCATION ---
    import warnings
    pure_kpts = np.array([frame["keypoints"] for frame in kp_seq])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        end = findPitchStart(pure_kpts) + 1
        start = max(0, end - 180)
    truncated_kp_seq = kp_seq[start:end]
    
    os.makedirs(savedir, exist_ok=True)
    base = os.path.splitext(os.path.basename(video))[0]
    np.save(os.path.join(savedir, f"{base}.npy"), np.array(truncated_kp_seq))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

pose_model = YOLO("yolov8x-pose.pt")
# Ensure best2.pt is uploaded to Colab directly before running!
glove_model = YOLO("best2.pt")
if torch.cuda.is_available():
    pose_model.to("cuda")
    glove_model.to("cuda")

videos = collectVideos("/content/drive/My Drive/Pitch-Tipping/Pitches")
print(f"Found {len(videos)} video(s) to process.")

In [ ]:
%%time
for video in videos:
    print(f"\n--- Processing: {video} ---")
    processVideo(pose_model, glove_model, video, "/content/drive/My Drive/Pitch-Tipping/Keypoints")